# Benchmark Comparison & Analysis

**Companion notebook** to `robustness_validation.ipynb`.  
Loads saved evaluation CSVs and computes SPY, 1/N, and MVO benchmarks.  
Run the evaluation notebook first to generate the required data.

## 1) Setup — Restore from Drive if needed

In [5]:
import os, sys
from pathlib import Path

EVAL_REPO_DIR = "/content/tcn_tape_vectorized_version_clean"
RUN_ID = "run10"
EPISODE_NO = "ep00404"
OUTPUT_DIR = Path("/content/robustness_results")
DRIVE_BACKUP = Path(f"/content/drive/MyDrive/robustness_{EPISODE_NO}_{RUN_ID}.zip")

# Auto-restore from Drive if local data is missing
if not OUTPUT_DIR.exists() or not (OUTPUT_DIR / f"{EPISODE_NO}_full_oos_daily.csv").exists():
    print("Local data not found — restoring from Drive backup...")
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    !unzip -q -o {DRIVE_BACKUP} -d {OUTPUT_DIR}
    print(f"[OK] Restored from: {DRIVE_BACKUP}")
else:
    print(f"[OK] Local data found at: {OUTPUT_DIR}")

# Also restore the repo + normalized CSV if needed
if not os.path.exists(EVAL_REPO_DIR):
    import subprocess
    subprocess.run(["git", "clone", "-b", "main",
                    "https://github.com/Dave-DKings/tcn_tape_vectorized_version.git",
                    EVAL_REPO_DIR], check=True)

# Restore normalized CSV from the evaluation zip if needed
EVAL_RESTORE_DIR = Path("/content/eval_restore")
norm_target = Path(EVAL_REPO_DIR) / "data" / "master_features_NORMALIZED.csv"
if not norm_target.exists():
    eval_zip = Path(f"/content/drive/MyDrive/tcn_tape_vectorized_{RUN_ID}.zip")
    if eval_zip.exists():
        import shutil
        EVAL_RESTORE_DIR.mkdir(parents=True, exist_ok=True)
        !unzip -q -o {eval_zip} -d {EVAL_RESTORE_DIR}
        candidates = list(EVAL_RESTORE_DIR.rglob("*normalized*.csv"))
        if candidates:
            norm_target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(max(candidates, key=lambda p: p.stat().st_mtime), norm_target)
            print(f"[OK] Restored normalized CSV to: {norm_target}")

os.chdir(EVAL_REPO_DIR)
print(f"[OK] Working dir: {os.getcwd()}")

## 2) Load evaluation data from CSV

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.optimize import minimize

EVAL_ASSET_UNIVERSE = ["MSFT","GOOGL","JPM","JNJ","XOM","PG","NEE","LIN","CAT","UNH"]

full_daily_df = pd.read_csv(OUTPUT_DIR / f"{EPISODE_NO}_full_oos_daily.csv", parse_dates=["date"])
det_results_df = pd.read_csv(OUTPUT_DIR / f"{EPISODE_NO}_deterministic_summary.csv")
all_stoch_df = pd.read_csv(OUTPUT_DIR / f"{EPISODE_NO}_stochastic_all_horizons.csv")
cross_horizon_df = pd.read_csv(OUTPUT_DIR / f"{EPISODE_NO}_cross_horizon_summary.csv")

stoch_by_horizon = {}
for hl in ["1yr","2yr","3yr","4yr","full"]:
    p = OUTPUT_DIR / f"{EPISODE_NO}_stochastic_{hl}.csv"
    if p.exists(): stoch_by_horizon[hl] = pd.read_csv(p)

print(f"[OK] Loaded evaluation data")
print(f"   full_daily_df: {full_daily_df.shape}")
print(f"   det_results_df: {det_results_df.shape}")
print(f"   all_stoch_df: {all_stoch_df.shape}")
print(f"   stoch horizons: {list(stoch_by_horizon.keys())}")

## 3) Load asset prices from normalized master

In [7]:
master_path = Path(EVAL_REPO_DIR) / "data" / "master_features_NORMALIZED.csv"
master_df = pd.read_csv(master_path)
master_df["Date"] = pd.to_datetime(master_df["Date"], utc=True, errors="coerce").dt.tz_localize(None)
master_df = master_df.dropna(subset=["Date"])

oos_start = full_daily_df["date"].min()
oos_end = full_daily_df["date"].max()
print(f"OOS window: {oos_start.date()} to {oos_end.date()}")

oos_master = master_df[(master_df["Date"] >= oos_start) & (master_df["Date"] <= oos_end)].copy()
oos_master = oos_master[oos_master["Ticker"].isin(EVAL_ASSET_UNIVERSE)]

if "LogReturn_1d" in oos_master.columns:
    ret_col = "LogReturn_1d"
elif "Return_1d" in oos_master.columns:
    ret_col = "Return_1d"
else:
    ret_candidates = [c for c in oos_master.columns if "return" in c.lower() and "1d" in c.lower()]
    ret_col = ret_candidates[0] if ret_candidates else None

if ret_col:
    asset_returns = oos_master.pivot_table(index="Date", columns="Ticker", values=ret_col)
    asset_returns = asset_returns[EVAL_ASSET_UNIVERSE].sort_index()
    asset_returns_simple = np.exp(asset_returns) - 1 if "Log" in ret_col else asset_returns
    print(f"[OK] Asset returns: {asset_returns_simple.shape} | col: {ret_col}")
else:
    print("[WARN] No return column found")

---
## 4) Benchmark 1: S&P 500 (SPY)

In [9]:
!pip install -q yfinance
import yfinance as yf
import numpy as np
import pandas as pd

spy_data = yf.download(
    "SPY",
    start=str(oos_start.date()),
    end=str((oos_end + pd.Timedelta(days=5)).date()),
    auto_adjust=True,
    progress=False,
)

# Normalize index tz
spy_data.index = pd.DatetimeIndex(spy_data.index).tz_localize(None)

# Robust close extraction (works for flat or MultiIndex columns)
if isinstance(spy_data.columns, pd.MultiIndex):
    if ("Close", "SPY") in spy_data.columns:
        spy_close = spy_data[("Close", "SPY")]
    elif ("Adj Close", "SPY") in spy_data.columns:
        spy_close = spy_data[("Adj Close", "SPY")]
    else:
        # fallback: first available Close-like column
        lvl0 = spy_data.columns.get_level_values(0)
        close_key = "Close" if "Close" in lvl0 else "Adj Close"
        spy_close = spy_data.xs(close_key, axis=1, level=0).iloc[:, 0]
else:
    close_col = "Close" if "Close" in spy_data.columns else "Adj Close"
    spy_close = spy_data[close_col]

spy_close = pd.to_numeric(spy_close, errors="coerce").dropna()
spy_returns = spy_close.pct_change().dropna()

eval_idx = pd.DatetimeIndex(full_daily_df["date"]).tz_localize(None)
spy_returns_aligned = spy_returns.reindex(eval_idx).dropna()

spy_wealth = (1 + spy_returns_aligned).cumprod() * 100_000
final_spy = float(spy_wealth.iloc[-1]) if len(spy_wealth) else np.nan

print(f"[OK] SPY: {len(spy_returns_aligned)}d | Return: {(final_spy/100_000 - 1)*100:.2f}%")


## 5) Benchmark 2: Equal-Weight (1/N)

In [10]:
n_assets = len(EVAL_ASSET_UNIVERSE)
ew_daily_returns = asset_returns_simple.mean(axis=1)
ew_returns_aligned = ew_daily_returns.reindex(pd.DatetimeIndex(eval_dates)).dropna()
ew_wealth = (1 + ew_returns_aligned).cumprod() * 100_000
print(f"[OK] 1/{n_assets}: {len(ew_returns_aligned)}d | Return: {(ew_wealth.iloc[-1]/100_000-1)*100:.2f}%")

## 6) Benchmark 3: MVO (Markowitz)

In [11]:
MVO_LOOKBACK = 126; MVO_REBAL_FREQ = 21; RF = 0.02/252

def solve_mvo(ret_win, rf=RF):
    mu, cov, n = ret_win.mean().values, ret_win.cov().values, len(ret_win.columns)
    def neg_sr(w): return -(w@mu-rf)/max(np.sqrt(w@cov@w),1e-10)
    res = minimize(neg_sr, np.ones(n)/n, method="SLSQP",
                   bounds=[(0,0.3)]*n, constraints=[{"type":"eq","fun":lambda w:w.sum()-1}])
    return res.x if res.success else np.ones(n)/n

dates_idx = asset_returns_simple.index
mvo_rets, w = [], np.ones(n_assets)/n_assets
for i in range(len(dates_idx)):
    mvo_rets.append(w @ asset_returns_simple.iloc[i].values)
    if i>0 and i%MVO_REBAL_FREQ==0 and i>=MVO_LOOKBACK:
        try: w = solve_mvo(asset_returns_simple.iloc[i-MVO_LOOKBACK:i])
        except: pass

mvo_series = pd.Series(mvo_rets, index=dates_idx, name="MVO")
mvo_returns_aligned = mvo_series.reindex(pd.DatetimeIndex(eval_dates)).dropna()
mvo_wealth = (1 + mvo_returns_aligned).cumprod() * 100_000
print(f"[OK] MVO: {len(mvo_returns_aligned)}d | Return: {(mvo_wealth.iloc[-1]/100_000-1)*100:.2f}%")

---
## 7) Full Benchmark Comparison Table

In [55]:
def compute_metrics(returns, name, cap=100_000):
    wealth = (1+returns).cumprod()*cap
    tot = (wealth.iloc[-1]/cap-1)*100; ny = len(returns)/252
    ann_r = ((1+tot/100)**(1/ny)-1)*100; ann_v = returns.std()*np.sqrt(252)*100
    sr = returns.mean()/returns.std()*np.sqrt(252) if returns.std()>0 else 0
    ds = returns[returns<0]; ds_std = ds.std() if len(ds)>0 else 1e-10
    sortino = returns.mean()/ds_std*np.sqrt(252) if ds_std>0 else 0
    dd = (wealth-wealth.cummax())/wealth.cummax(); mdd = abs(dd.min())*100
    calmar = ann_r/mdd if mdd>0 else 0
    wr = (returns>0).mean()*100
    var95 = np.percentile(returns,5)*100
    cvar95 = returns[returns<=np.percentile(returns,5)].mean()*100
    return {"Strategy":name,"Ann. Return (%)":round(ann_r,2),"Ann. Vol (%)":round(ann_v,2),
            "Sharpe":round(sr,3),"Sortino":round(sortino,3),"Max DD (%)":round(mdd,2),
            "Calmar":round(calmar,3),"Win Rate (%)":round(wr,1),
            "VaR 95% (%)":round(var95,3),"CVaR 95% (%)":round(cvar95,3),"Total Ret (%)":round(tot,2)}

agent_returns = full_daily_df["daily_return"].dropna()
agent_returns.index = pd.DatetimeIndex(full_daily_df["date"].iloc[:len(agent_returns)])

benchmark_df = pd.DataFrame([
    compute_metrics(agent_returns, f"TAPE-TCN ({EPISODE_NO})"),
    compute_metrics(spy_returns_aligned, "S&P 500 (SPY)"),
    compute_metrics(ew_returns_aligned, f"Equal-Weight (1/{n_assets})"),
    compute_metrics(mvo_returns_aligned, "MVO (Markowitz)"),
]).set_index("Strategy")

print("="*80); print("BENCHMARK COMPARISON — FULL OOS"); print("="*80)
display(benchmark_df)
benchmark_df.to_csv(OUTPUT_DIR / f"{EPISODE_NO}_benchmark_comparison.csv")

NameError: name 'full_daily_df' is not defined

---
## 8) Cumulative Wealth Curve (Primary Paper Figure)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), height_ratios=[3,1],
                                sharex=True, gridspec_kw={"hspace":0.05})
fig.suptitle("Out-of-Sample Performance: TAPE-TCN vs. Benchmarks", fontsize=15, fontweight="bold")

agent_wealth = full_daily_df.set_index("date")["portfolio_value"]
ax1.plot(agent_wealth.index, agent_wealth.values, label=f"TAPE-TCN ({EPISODE_NO})", lw=2.0, color="#1f77b4", zorder=5)
ax1.plot(spy_wealth.index, spy_wealth.values, label="S&P 500 (SPY)", lw=1.5, color="#ff7f0e", alpha=0.8)
ax1.plot(ew_wealth.index, ew_wealth.values, label=f"Equal-Weight (1/{n_assets})", lw=1.5, color="#2ca02c", alpha=0.8)
ax1.plot(mvo_wealth.index, mvo_wealth.values, label="MVO (Markowitz)", lw=1.5, color="#d62728", alpha=0.8)

for ds, lb in {"2020-03-23":"COVID\nBottom","2022-01-03":"Rate\nHikes","2023-03-10":"Banking\nCrisis"}.items():
    d = pd.to_datetime(ds)
    if d >= agent_wealth.index.min() and d <= agent_wealth.index.max():
        ax1.axvline(x=d, color="gray", ls=":", alpha=0.5)
        ax1.text(d, ax1.get_ylim()[1]*0.95, lb, ha="center", fontsize=7, color="gray")

ax1.set_ylabel("Portfolio Value ($)", fontsize=12)
ax1.legend(loc="upper left", fontsize=10); ax1.grid(True, alpha=0.3)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"${x:,.0f}"))

agent_dd = full_daily_df.set_index("date")["drawdown"]*100
spy_dd = ((spy_wealth-spy_wealth.cummax())/spy_wealth.cummax())*100
ax2.fill_between(agent_dd.index, agent_dd.values, 0, alpha=0.4, color="#1f77b4", label="TAPE-TCN")
ax2.plot(spy_dd.index, spy_dd.values, lw=1.0, color="#ff7f0e", alpha=0.7, label="SPY")
ax2.set_ylabel("Drawdown (%)"); ax2.set_xlabel("Date")
ax2.legend(loc="lower left", fontsize=9); ax2.grid(True, alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

plt.tight_layout()
fig.savefig(OUTPUT_DIR / f"{EPISODE_NO}_wealth_curve_benchmarks.png", dpi=200, bbox_inches="tight")
plt.show()

## 9) Rolling Sharpe Comparison

In [ ]:
ROLL = 126
def roll_sr(r, w=ROLL): return (r.rolling(w).mean()/r.rolling(w).std())*np.sqrt(252)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(roll_sr(agent_returns), label="TAPE-TCN", lw=1.5, color="#1f77b4")
ax.plot(roll_sr(spy_returns_aligned), label="SPY", lw=1.2, color="#ff7f0e", alpha=0.7)
ax.plot(roll_sr(ew_returns_aligned), label="1/N", lw=1.2, color="#2ca02c", alpha=0.7)
ax.plot(roll_sr(mvo_returns_aligned), label="MVO", lw=1.2, color="#d62728", alpha=0.7)
ax.axhline(y=0, color="black", ls="-", alpha=0.3)
ax.axhline(y=1, color="green", ls="--", alpha=0.3, label="Sharpe=1")
ax.set_ylabel("Rolling 6-Month Sharpe"); ax.set_title(f"Rolling {ROLL}-Day Sharpe Ratio")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
fig.savefig(OUTPUT_DIR / f"{EPISODE_NO}_rolling_sharpe.png", dpi=150, bbox_inches="tight")
plt.show()

## 10) Outperformance Consistency

In [ ]:
excess = (agent_returns - spy_returns_aligned.reindex(agent_returns.index)).dropna()
roll_excess = excess.rolling(252).sum()*100

fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(roll_excess.index, roll_excess.values, 0,
                where=roll_excess.values>=0, color="green", alpha=0.4, label="Outperforming SPY")
ax.fill_between(roll_excess.index, roll_excess.values, 0,
                where=roll_excess.values<0, color="red", alpha=0.4, label="Underperforming SPY")
ax.axhline(y=0, color="black", lw=0.8)
ax.set_ylabel("Rolling 1yr Excess Return (%)")
ax.set_title("TAPE-TCN Excess Return vs. S&P 500")
pct = (roll_excess.dropna()>0).mean()*100
ax.legend(title=f"Outperformance rate: {pct:.0f}%", fontsize=9)
ax.grid(True, alpha=0.3)
fig.savefig(OUTPUT_DIR / f"{EPISODE_NO}_excess_return_spy.png", dpi=150, bbox_inches="tight")
plt.show()

## 11) Transaction Cost Sensitivity

In [ ]:
if "daily_turnover" in full_daily_df.columns:
    turnover = full_daily_df["daily_turnover"].values
else:
    w_cols = [c for c in full_daily_df.columns if c.startswith("w_")]
    if w_cols:
        wm = full_daily_df[w_cols].values
        turnover = np.concatenate([[0], np.sum(np.abs(np.diff(wm, axis=0)), axis=1)])
    else:
        turnover = np.zeros(len(full_daily_df))

gross = full_daily_df["daily_return"].values
cost_results = []
for bps in [0, 5, 10, 20, 50]:
    net = gross - turnover*(bps/10000)
    net_s = pd.Series(net).dropna()
    sr = net_s.mean()/net_s.std()*np.sqrt(252) if net_s.std()>0 else 0
    tot = ((1+net_s).prod()-1)*100
    ar = ((1+tot/100)**(252/len(net_s))-1)*100
    cost_results.append({"Cost (bps)":bps, "Net Sharpe":round(sr,3),
                         "Net Ann. Return (%)":round(ar,2), "Net Total Return (%)":round(tot,2)})

cost_df = pd.DataFrame(cost_results)
print("\nTRANSACTION COST SENSITIVITY:")
display(cost_df)
cost_df.to_csv(OUTPUT_DIR / f"{EPISODE_NO}_cost_sensitivity.csv", index=False)

## 12) Save all outputs + Drive backup

In [ ]:
benchmark_daily = pd.DataFrame({
    "date": full_daily_df["date"],
    "agent_value": full_daily_df["portfolio_value"],
    "agent_return": full_daily_df["daily_return"],
})
for nm, wl, rt in [("spy",spy_wealth,spy_returns_aligned),
                    ("ew",ew_wealth,ew_returns_aligned),
                    ("mvo",mvo_wealth,mvo_returns_aligned)]:
    benchmark_daily[f"{nm}_value"] = wl.reindex(pd.DatetimeIndex(eval_dates)).values[:len(benchmark_daily)]
    benchmark_daily[f"{nm}_return"] = rt.reindex(pd.DatetimeIndex(eval_dates)).values[:len(benchmark_daily)]

benchmark_daily.to_csv(OUTPUT_DIR / f"{EPISODE_NO}_all_daily_with_benchmarks.csv", index=False)
print(f"[OK] Saved combined daily: {benchmark_daily.shape}")

import zipfile
with zipfile.ZipFile(DRIVE_BACKUP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fp in OUTPUT_DIR.rglob("*"):
        if fp.is_file(): zf.write(fp, fp.relative_to(OUTPUT_DIR))
print(f"[OK] Updated Drive backup: {DRIVE_BACKUP}")